# 模块2 第1节：通过离线评估建立基准性能

目前我们有一个针对TechHub的MVP客服代理，能够回答订单相关问题、提供产品信息，并解释 store政策。然而，在将它展示给客户之前，我们需要先建立对其期望功能的信心。

在本模块中，我们将学习如何运行离线评估来确定基准性能，然后通过以评估驱动的方式系统地对其进行改进。

**什么是离线评估？**

评估是一个至关重要的、持续的过程，它允许我们定量衡量我们的应用的性能，识别需要改进的地方，并随着时间的推移可靠地进化我们的系统。

<div align="center">
    <img src="../../images/offline_eval_process.png">
</div>

离线评估过程由几个组成部分组成：

1. 数据集 - 一组经过精选的具有代表性的示例，每个示例包括：
    - 系统的输入
    - 参考输出（参考答案），展示了预期的高质量结果应该是什么样子
2. 应用程序 - 我们打算进行评估的LLM系统。我们将输入提供给我们的示例输入，并收集系统的实际输出。
3. 评价器 - 通过比较输入、输出和参考输出来量化某些方面的性能的一些函数

**构建良好评估设置的关键点**

在开始创建一个新的评估suite时，通常建议：

- 收集一组代表你系统核心功能及其应处理的场景的小批量标签示例。
- 借助领域专家的知识，确保这些示例具有代表性且准确。
- 选择少量简单的指标。在实践中，二元评价指标迫使我们有更清晰的思考、更一致的标注，并且在分析和迭代系统时更容易/更快地理解。

从大量样本或许多复杂指标开始会导致难以检查并深入理解系统行为，从而很快陷入分析 paralysis。

让我们看看如何在LangSmith中对我们的TechHub代理进行离线评估，以建立基准性能！

#### 安装
```markdown
# 安装

## 前置准备
确保你的环境配置已正确设置，包括：

- **Python**：安装最新版本的 Python 并启用内核。
- **Jupyter**：安装并配置 Jupyter Notebook 或 JupyterLab。

## 安装步骤
按照以下步骤逐步安装所需组件：

1. **安装依赖项**：
   ```bash
   pip install -r requirements.txt
   ```

2. **安装主组件**：
   ```bash
   git clone https://github.com/your-org LangChain
   cd LangChain
   python setup.py install
   ```

3. **配置环境变量**：
   根据项目需求设置必要的环境变量，例如：
   ```python
   import os
   os.environ['PATH'] = 'path/to/tool:$PATH'
   ```

4. **运行服务**：
   启动并监控相关服务：
   ```bash
   python main.py
   ```
   检查服务状态和日志，确保一切正常。

5. **测试集成**：
   进行必要的测试以确保各组件集成顺利：
   ```python
   from your_module import your_function
   your_function()
   ```

6. **部署到云服务**（可选）：
   如果需要，可以使用 AWS、GCP 或 Azure 等平台进行部署。

7. **启动服务**：
   根据部署结果启动相关服务。
   ```bash
   gunicorn main:app --bind 0.0.0.0:8000
   ```

8. **监控和维护**：
   定期检查系统性能并进行必要的维护。

9. **文档编写**：
   使用 sphinx 或 rst 等工具编写项目文档。
   ```bash
   Sphinx:
   make html
   ```

10. **部署到生产环境**：
    在正式环境中运行服务并进行测试。

11. **持续集成/交付（CI/CD）**：
    配置 CI/CD管道以自动化测试和部署流程。

12. **性能优化**：
    根据测试结果优化系统性能。
   ```bash
   pip install -U pip
   ```

13. **更新依赖项**：
    定期更新所有依赖项以获取最新功能和安全补丁。

14. **部署到生产环境**（可选）：
    根据项目需求选择是否在生产环境中运行服务。
   ```bash
   python main.py --production
   ```

15. **监控和维护**：
    在生产环境中持续监控系统性能并进行必要的维护。

16. **文档编写**：
    使用 sphinx 或 rst 等工具编写项目文档。
   ```bash
   Sphinx:
   make html
   ```

17. **部署到生产环境**（可选）：
    根据项目需求选择是否在生产环境中运行服务。
   ```bash
   python main.py --production
   ```

18. **监控和维护**：
    在生产环境中持续监控系统性能并进行必要的维护。

19. **文档编写**：
    使用 sphinx 或 rst 等工具编写项目文档。
   ```bash
   Sphinx:
   make html
   ```

20. **部署到生产环境**（可选）：
    根据项目需求选择是否在生产环境中运行服务。
   ```bash
   python main.py --production
   ```
```

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

## 1. 构建一个具有代表性的示例集

在我们使用场景中，我们与TechHub客户服务团队合作，创建了10个真实示例。每个示例包含：

- **输入**：顾客的问题
- **输出**：预期的正确答案（即真实值）
- **元数据**：客户支持团队使用的分类问题类型

该数据集结构允许我们评估代理的“端到端”性能，通常称为[最终回答评估](https://docs.langchain.com/langsmith/evaluation-approaches#evaluating-an-agent%E2%80%99s-final-response)。

<div align="center">
#     <img src="../../images/final-response-eval.png" width="800">
</div>

让我们加载并探索该数据集：

In [ ]:
import json
from pathlib import Path
from pprint import pprint

# Load the dataset from JSON
dataset_path = Path("baseline_dataset.json")

with open(dataset_path, "r") as f:
    examples = json.load(f)

In [ ]:
pprint(examples[0])

## 2. 在 LangSmith 中创建一个数据集

注意：我们将示例转换为 `messages` 格式，以便它们在 LangSmith 中原样存储，并且具有使用这些示例所需的结构，以便在调用我们的代理时能够使用。

In [ ]:
import uuid
from langsmith import Client

client = Client()

dataset_name = f"techhub-baseline-eval-{uuid.uuid4()}"
dataset_description = "Representative set of customer support questions and answers curated by our support team"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description=dataset_description,
)

client.create_examples(
    dataset_id=dataset.id,
    inputs=[
        {"messages": [{"role": "user", "content": ex["inputs"]["question"]}]}
        for ex in examples
    ],
    outputs=[
        {"messages": [{"role": "assistant", "content": ex["outputs"]["answer"]}]}
        for ex in examples
    ],
    metadata=[ex["metadata"] for ex in examples],
)

print(f"Dataset in LangSmith: {dataset.url}")

## 3. 初始化我们要评估的代理

我们将使用我们在模块1第4节中构建的带有HITL验证的监督员代理进行初始化。

In [ ]:
from IPython.display import Image
from agents.supervisor_hitl_agent import create_supervisor_hitl_agent

agent = create_supervisor_hitl_agent()

display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# 快速测试

## 测试你的知识！

### 问题1：什么是LangChain？

什么是LangChain？LangChain是一个构建具有语言理解与处理能力的应用程序的框架。

### 问题2：在LangGraph的语境中，“Tool Calling”是什么意思？

在LangGraph的语境中，“Tool Calling”指的是调用外部工具或服务以执行特定任务的过程。

In [ ]:
import uuid

thread_id = uuid.uuid4()
config = {"configurable": {"thread_id": thread_id}}

t = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's your return policy for opened electronics?",
            }
        ]
    },
    config=config,
)

In [ ]:
t["messages"][-1].pretty_print()

## 4. 定义我们的评估器

评估器是衡量您应用在特定示例上表现程度的函数。

我们将从两个简单的评估器开始：`correctness` 和 `total_tool_calls`

### 评估器#1：正确性

一个使用LLM-as-a-Judge来判断代理输出是否"正确"的评估器，即比较代理输出与参考输出（即 ground truth 输出）以确定其正确性。

> 注意：我们手动定义了LLM-as-a-Judge评估器 below for clarity, 但你可以通过 openevals 库实现相同的目标，代码行数更少。

In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from config import DEFAULT_MODEL


CORRECTNESS_PROMPT = """You are an expert data labeler evaluating model outputs for correctness.

Your task is to assign a boolean score based on the following rubric:

<Rubric>
  A correct answer (True):
  - Provides accurate and complete information
  - Contains no factual errors
  - Addresses all parts of the question
  - Is logically consistent
</Rubric>

<Instructions>
  - Carefully read the input and output
  - Compare the output to the reference_output
  - Check for factual accuracy and completeness
  - Focus on correctness of information rather than style or verbosity differences
  - Return a boolean score (True if correct, False if incorrect), not a string
</Instructions>

<Note>
- It's ok if the ouput provides additional information that is not directly included in the reference output
- The output is just the final output from an agent invocation, so it will not include all the intermediate steps or tool calls, this is ok.
</Note>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<reference_outputs>
{reference_outputs}
</reference_outputs>
"""


# For structured LLM output
class CorrectnessScore(BaseModel):
    reasoning: str = Field(..., description="A concise reasoning for the score")
    score: bool = Field(
        ..., description="True if the output is correct, False if incorrect."
    )


# Create a structured LLM
correctness_evaluator_llm = init_chat_model(model=DEFAULT_MODEL).with_structured_output(
    CorrectnessScore
)


# Define the evaluator function
def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Evaluate the correctness of the output against the reference output."""

    formatted_prompt = CORRECTNESS_PROMPT.format(
        inputs=inputs, outputs=outputs, reference_outputs=reference_outputs
    )

    eval_result = correctness_evaluator_llm.invoke(formatted_prompt)

    # return a dictionary with the format the evaluator expects
    return {
        "key": "correctness",
        "score": eval_result.score,
        "comment": eval_result.reasoning,
    }

**使用 `openevals` 短语：** 手动实现上述内容有助于理解其内部工作机制。在实际应用中，`openevals` 将所有这些内容——结构化输出模型、LLM 调用和返回格式——封装成一个工厂函数：

```python
from openevals.llm import create_llm_as_judge

correctness_evaluator = create_llm_as_judge(
    model=DEFAULT_MODEL,
    prompt=CORRECTNESS_PROMPT,  # 使用与上文相同的提示模板
    feedback_key="correctness",
)
```

返回的 `correctness_evaluator` 是一个 drop-in 替换方案——将其直接传递给 `client.evaluate()` 方法，与其他方式完全一致。

#### 目标函数

目标函数用于指定从我们的数据集中如何处理`输入`以生成我们希望评估的`输出`。在这种情况下，它只是将`输入`通过我们的代理来产生响应消息即可。

In [ ]:
import uuid


def target_function(inputs: dict) -> dict:
    """Target function that runs our agent to get outputs for evaluation."""

    thread_id = uuid.uuid4()
    config = {"configurable": {"thread_id": thread_id}}

    result = agent.invoke(
        inputs,
        config=config,
    )

    return {
        "messages": [{"role": "assistant", "content": result["messages"][-1].content}]
    }

现在，我们将在我们的数据集中从一个示例中测试 `correctness` 评估器，以了解整个流程的运作方式

In [ ]:
# get an example from our dataset
example = next(
    client.list_examples(dataset_id=dataset.id, metadata={"example_number": 1})
)
pprint(example.inputs)

In [ ]:
# run the example inputs through our target function
output = target_function(example.inputs)
pprint(output)

In [ ]:
# score the agent output against the reference output
correctness_score = correctness_evaluator(
    inputs=example.inputs, outputs=output, reference_outputs=example.outputs
)
pprint(correctness_score)

### 评估器 #2：总工具调用次数

一个不依赖于地面真理参考的评估器，但它是一个很好的指标，可以帮助揭示代理模式并突出显示效率问题

In [ ]:
from langsmith.schemas import Run


def count_total_tool_calls_evaluator(run: Run) -> dict:
    """
    Count total tool calls across the entire run (supervisor + sub-agents).

    Returns a single 'score' metric with the total count.
    Use for tracking efficiency: fewer calls = more efficient.
    """

    def traverse_runs(run_obj: Run) -> int:
        """Recursively count all tool-type runs in the tree."""
        count = 0

        # Count this run if it's a tool execution
        if run_obj.run_type == "tool":
            count = 1

        # Recursively count child runs
        if hasattr(run_obj, "child_runs") and run_obj.child_runs:
            for child in run_obj.child_runs:
                count += traverse_runs(child)

        return count

    total_tools = traverse_runs(run)

    return {"key": "total_tool_calls", "score": total_tools}

这个评估器不依赖参考输出来生成评分，而是通过目标函数在给定示例上的执行 metadata 来计算分数。这种机制在 LangSmith SDK 中灵活实现，只需将一个 Run 对象传递给评估器。

> 注意：请参阅 [文档页面](https://docs.langchain.com/langsmith/code-evaluator#evaluator-args) 以获取 evaluator 函数接受的所有参数列表

让我们通过一个例子来说明这个机制如何运作。

In [ ]:
# get a recent sample run from our project
runs = client.list_runs(
    project_name="langsmith-agent-lifecycle-workshop",  # Your project
    filter="""and(eq(is_root, true), eq(name, "supervisor_hitl_agent"))""",  # get a LangGraph (i.e. supervisor) run
    limit=1,
)
run = next(runs)

# Fetch the complete run with all children
full_run = client.read_run(run.id, load_child_runs=True)

In [ ]:
# we can inspect the full run metadata
# vars(full_run)

# or just look at the child runs
vars(full_run).get("child_runs")

In [ ]:
# now lets pass the run to our evaluator
count_total_tool_calls_evaluator(full_run)

## 5. 在全量数据集上运行实验

我们现在可以利用我们的 target_function 和两个评价器，程序性地在我们数据集的每个示例上运行Offline评估——这在LangSmith中被称为"实验"。

In [ ]:
# Disable parallelism warnings from the tokenizers library to keep notebook output clean
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Run the experiment
results = client.evaluate(
    target_function,
    data=dataset_name,
    evaluators=[correctness_evaluator, count_total_tool_calls_evaluator],
    experiment_prefix="baseline-eval",
    description="Evaluate the final answer correctness and total tool calls of our agent on the baseline dataset",
    max_concurrency=5,
)

## 6. 错误分析在LangSmith界面中

现在，我们通过上述链接来分析LangSmith界面的基础性能。